In [113]:
from pandas import read_csv, DataFrame, concat
from numpy import percentile
from sklearn.preprocessing import StandardScaler

In [114]:
dataframe_raw = read_csv('../data/raw/bcw_data_raw.csv', )
dataframe_raw.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


# Data analysis
There are 569 observations and 32 columns, with 1 qualitative and 31 quantitative columns. 
For these columns :  
- 'Diagnosis' (M = malignant, B = benign)  

Ten real-valued features are computed for each cell nucleus :  
- 'radius' (mean of distances from center to points on the perimeter)
- 'texture' (standard deviation of gray-scale values)
- 'perimeter'
- 'area'
- 'smoothness' (local variation in radius lengths)
- 'compactness' (perimeter^2 / area - 1.0)
- 'concavity' (severity of concave portions of the contour)
- 'concave' points (number of concave portions of the contour)
- 'symmetry' 
- 'fractal' dimension ("coastline approximation")  
***
There aren't any null values.
***
## Drop 
### Drop columns
First if columns are not relevant to analyse or to train prediction models they must be removed.  

In [115]:
COLUMNS_TO_DROP = ['Unnamed: 32', 'id']
dataframe_with_best_columns = dataframe_raw.drop(columns=COLUMNS_TO_DROP)

### Drop observations (outliers)
Thanks to Tukey's method with percentile, we remove outliers.

In [116]:
def drop_outliers(
    dataframe: DataFrame, columns: list[str], percent: int = 75
) -> DataFrame:
    """
    With Tukey's method to find outliers, we change quartile by input percentile & drop outliers
    """
    for column in columns:
        if dataframe[column].dtype not in ["int64", "float64"]:
            continue
        first_quartile = percentile(dataframe[column], 100 - percent)
        third_quartile = percentile(dataframe[column], percent)
        step = 1.5 * (third_quartile - first_quartile)

        dataframe.drop(
            dataframe.loc[
                ~(
                    (dataframe[column] >= first_quartile - step)
                    & (dataframe[column] <= third_quartile + step)
                ),
                column,
            ].index,
            inplace=True,
        )
    return dataframe

dataframe_droped = drop_outliers(dataframe_with_best_columns, dataframe_with_best_columns.columns, 95)
dataframe_droped.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 558 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                558 non-null    object 
 1   radius_mean              558 non-null    float64
 2   texture_mean             558 non-null    float64
 3   perimeter_mean           558 non-null    float64
 4   area_mean                558 non-null    float64
 5   smoothness_mean          558 non-null    float64
 6   compactness_mean         558 non-null    float64
 7   concavity_mean           558 non-null    float64
 8   concave points_mean      558 non-null    float64
 9   symmetry_mean            558 non-null    float64
 10  fractal_dimension_mean   558 non-null    float64
 11  radius_se                558 non-null    float64
 12  texture_se               558 non-null    float64
 13  perimeter_se             558 non-null    float64
 14  area_se                  5

As we can see, with a percentile of 95%, there are 11 outliers. We think it's relevant to remove them with such a small amount but we guess a more important impact.  
*** 
## New dataframe
Now we have 31 columns (-1) with 558 observations (-11). It's a small but still relevant dataframe. 
# Data preprocessing
Now let's preprocess with :  
- target_column to boolean
- standardisation of numerical columns
 

In [117]:
dataframe, target_values = dataframe_droped.drop(columns=['diagnosis']), dataframe_droped['diagnosis']

scaler = StandardScaler().set_output(transform='pandas')
dataframe_preprocess = scaler.fit_transform(dataframe)

target_values = target_values.map({'B':0, 'M':1})

final_dataframe = concat([dataframe_preprocess, target_values], axis=1)
final_dataframe.to_csv('../data/preprocess/bcw_data_preprocess.csv')